# ==============================================================================
# 🍷 SOMMELIER SENSORY GRADING: WINE QUALITY VIA MULTICLASS SUPPORT VECTOR CLASSIFIER
# ==============================================================================
### Core Principles:
Grading wine sensory tiers based on physiochemical attributes (acidity, volatile acidity, residual sugar, sulfur dioxide, pH, alcohol).
SVM natively supports multiclass classification via **One-vs-One (OvO)**:
For $K$ classes, $\frac{K(K-1)}{2}$ binary hyperplanes are trained, voting for the highest scoring grade.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Sommelier Grading Environment ready.")


✅ STEP 1: Sommelier Grading Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle WineQT dataset (1,143 wine chemical profiles).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/wine_quality/WineQT.csv'
df = pd.read_csv(data_path)

if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# Collapse into 3 standard commercial tiers: Low (quality <= 5), Medium (quality == 6), High (quality >= 7)
def categorize_quality(q):
    if q <= 5: return 0  # Low / Table
    elif q == 6: return 1 # Medium / Commercial
    else: return 2       # High / Reserve

df['quality_tier'] = df['quality'].apply(categorize_quality)

print(f"📊 Dataset Shape: {df.shape[0]} wines, {df.shape[1]} attributes")
print(f"Commercial Tier Breakdown:\n{df['quality_tier'].value_counts(normalize=True).round(4)*100}%")
df.head(3)


📊 Dataset Shape: 1143 wines, 13 attributes
Commercial Tier Breakdown:
quality_tier
0    45.67
1    40.42
2    13.91
Name: proportion, dtype: float64%


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality,quality_tier
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating chemical features from target `quality_tier`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['quality', 'quality_tier'])
y = df['quality_tier']

print(f"Chemical Features: {X.shape[1]}")
print(f"Missing values: {X.isnull().sum().sum()}")


Chemical Features: 11
Missing values: 0


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 stratified split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Wines: {X_train.shape[0]}")
print(f"Test Wines : {X_test.shape[0]}")


Train Wines: 914
Test Wines : 229


## ⚔️ STEP 7 & 8: POLYNOMIAL VS RBF MULTICLASS DUEL
Testing degree-3 polynomial chemistry curves against RBF Gaussian manifolds.


In [5]:
# STEP 7 & 8: POLYNOMIAL VS RBF
pipe_poly = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='poly', degree=3, decision_function_shape='ovo', random_state=42))
])
pipe_poly.fit(X_train, y_train)

pipe_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', decision_function_shape='ovo', random_state=42))
])
pipe_rbf.fit(X_train, y_train)

acc_poly = accuracy_score(y_test, pipe_poly.predict(X_test))
acc_rbf = accuracy_score(y_test, pipe_rbf.predict(X_test))

print("="*65)
print(f"Polynomial (deg=3) Accuracy: {acc_poly*100:.2f}%")
print(f"RBF Gaussian Kernel Accuracy: {acc_rbf*100:.2f}%")
print("="*65)


Polynomial (deg=3) Accuracy: 65.07%
RBF Gaussian Kernel Accuracy: 67.69%


## 🎯 STEP 9: TUNING MULTICLASS HYPERPARAMETERS VIA 5-FOLD CV
Optimizing $C$ and $\gamma$.


In [6]:
# STEP 9: GRID SEARCH
param_grid = {
    'svc__C': [1, 5, 10, 20],
    'svc__gamma': ['scale', 0.05, 0.1, 0.5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_rbf, param_grid=param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Sommelier Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Accuracy: {grid.best_score_*100:.2f}%")


🏆 Best Sommelier Hyperparameters: {'svc__C': 10, 'svc__gamma': 0.05}
🎯 Best 5-Fold CV Accuracy: 67.17%


## 📊 STEP 10 & 11: MULTICLASS EVALUATION & CONFUSION MATRIX
Reviewing classification accuracy across Low, Medium, and High reserve tiers.


In [7]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("📋 SOMMELIER COMMERCIAL GRADING REPORT:")
print(classification_report(y_test, y_pred, target_names=['Low / Table', 'Medium / Standard', 'High / Reserve']))

print(f"🎯 Overall Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 Cohen's Kappa   : {cohen_kappa_score(y_test, y_pred):.4f}")

print(f"\nMulticlass Confusion Matrix:\n{cm}")


📋 SOMMELIER COMMERCIAL GRADING REPORT:
                   precision    recall  f1-score   support

      Low / Table       0.75      0.73      0.74       105
Medium / Standard       0.62      0.66      0.64        92
   High / Reserve       0.57      0.50      0.53        32

         accuracy                           0.67       229
        macro avg       0.65      0.63      0.64       229
     weighted avg       0.67      0.67      0.67       229

🎯 Overall Accuracy: 67.25%
🎯 Cohen's Kappa   : 0.4587

Multiclass Confusion Matrix:
[[77 25  3]
 [22 61  9]
 [ 3 13 16]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & CELLAR SMOKE TEST
Deploying pipeline and validating winery grading latency.


In [8]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/wine_quality_svc_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_wine = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_tier = loaded_pipe.predict(sample_wine)[0]
latency_ms = (time.perf_counter() - t0) * 1000

tier_names = {0: 'Low / Table Wine', 1: 'Medium / Standard Vintage', 2: 'High / Reserve Quality'}
print(f"\n🍇 LIVE WINERY BOTTLING AUDIT:")
print(f"   Automated Wine Grade: 🍾 {tier_names[pred_tier]}")
print(f"   Grading Latency     : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/wine_quality_svc_pipeline.joblib (27,737 bytes)

🍇 LIVE WINERY BOTTLING AUDIT:
   Automated Wine Grade: 🍾 Medium / Standard Vintage
   Grading Latency     : 1.584 ms (SLA < 2.0ms: PASS)
